In [ ]:
import geopandas as gpd
import pandas as pd
from tqdm import tqdm

# Load shapefiles
grid_shp = gpd.read_file("/home/stagiaire/Téléchargements/TE3/mesh.shp")
vegetation = gpd.read_file("/home/stagiaire/Téléchargements/TE3/V.shp")
ghsl = gpd.read_file("/home/stagiaire/Téléchargements/TE3/GHSL_FIN_S1000.shp")
favelas = gpd.read_file("/home/stagiaire/Téléchargements/TE3/F.shp")
osm = gpd.read_file("/home/stagiaire/Téléchargements/TE3/open_street_map.shp")

# Ensure CRS consistency
if favelas.crs != grid_shp.crs:
    favelas = favelas.to_crs(grid_shp.crs)

# Add a column for the cell surface area
grid_shp['cell_surface'] = grid_shp.geometry.area

# Function to compute intersection proportions
def compute_intersection_proportions(grid, dataset, label):
    intersections = gpd.sjoin(grid, dataset, how="left", predicate="intersects").reset_index()
    tqdm.pandas(desc=f"Computing proportions for {label}")
    intersections['intersection_surface'] = intersections.progress_apply(
        lambda row: row['geometry'].intersection(dataset.loc[row['index_right'], 'geometry']).area / row['cell_surface'] 
        if pd.notnull(row['index_right']) else 0,
        axis=1
    )
    return intersections.groupby('index').agg({'intersection_surface': 'sum'}).rename(columns={'intersection_surface': label})

# Compute proportions for each dataset
vegetation_agg = compute_intersection_proportions(grid_shp, vegetation, 'vegetation')
ghsl_agg = compute_intersection_proportions(grid_shp, ghsl, 'ghsl')
favelas_agg = compute_intersection_proportions(grid_shp, favelas, 'favelas')
osm_agg = compute_intersection_proportions(grid_shp, osm, 'osm')


In [ ]:
grille_shp = grille_shp.reset_index().merge(vegetation_agg, on='index', how='left')
grille_shp = grille_shp.merge(ghsl_agg, on='index', how='left')
grille_shp = grille_shp.merge(favelas_agg, on='index', how='left')
grille_shp = grille_shp.merge(osm_agg, on='index', how='left')

grille_shp['vegetation'] = grille_shp['vegetation'].fillna(0)
grille_shp['ghsl'] = grille_shp['ghsl'].fillna(0)
grille_shp['favelas'] = grille_shp['favelas'].fillna(0)
grille_shp['osm'] = grille_shp['osm'].fillna(0)


grille_shp.to_file("grille.shp")
